# Efeito Joule-Thomson numa válvula globo: gás natural, gás real e o seu CFD

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PINNeAPPle-Labs/PINNeAPPle/blob/main/notebooks/colab/07_valvula_joule_thomson.ipynb)

Quando o gás natural passa por uma válvula, a pressão cai e o gás esfria. Isso é o **efeito Joule-Thomson**. A
válvula não troca calor nem realiza trabalho, então a **entalpia se conserva** através dela ($h_2 = h_1$). A
temperatura de saída sai desse balanço, e o coeficiente que liga as duas é

$$\mu_{JT} = \left(\frac{\partial T}{\partial p}\right)_h .$$

Para um **gás ideal**, a entalpia depende só da temperatura, então $\mu_{JT} = 0$ e **não há resfriamento nenhum**.
Isso explica o caso mais comum de "CFD que não mostra o efeito": a simulação usa gás ideal (`perfectGas` no
OpenFOAM, *ideal gas* no Fluent ou no CFX). Com o gás real, uma queda de 70 para 20 bar esfria o gás natural uns
25 K.

Este notebook usa a biblioteca aberta **PINNeAPPle** (`pinneapple_systems.process_components.joule_thomson`) para:

1. ler o caso escrito em palavras;
2. calcular a temperatura de saída com quatro equações de estado: GERG-2008 (a referência para gás natural),
   Peng-Robinson, SRK e gás ideal;
3. traçar o caminho isentálpico e o $\mu_{JT}$ ao longo dele;
4. checar a vazão da válvula (IEC 60534-2-1) e se ela está bloqueada (*choked*);
5. comparar com a temperatura que o seu CFD deu e receber uma nota VeriPhysics, com o diagnóstico;
6. ver a sensibilidade à pressão de saída, à temperatura de entrada e à composição;
7. gerar a geometria da válvula com o PINNeAPPle e rodar o **CFD no OpenFOAM pelo PINNeAPPle**, com gás ideal e
   com Peng-Robinson, e comparar com o estrangulamento.

As propriedades vêm do CoolProp (GERG-2008 e as cúbicas). Roda em CPU, em segundos; não precisa de GPU.

In [ ]:
# Instalação: no Colab (ou onde faltar), o PINNeAPPle do PyPI, o CoolProp e o gmsh. Numa versão do PyPI que ainda
# não traz os módulos de Joule-Thomson e do CFD da válvula, os arquivos que eles usam vêm do repositório no GitHub.
import importlib, importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

if importlib.util.find_spec("CoolProp") is None:
    pip("CoolProp>=6.6")
if importlib.util.find_spec("pinneapple_systems") is None:
    pip("pinneapple>=0.7.0")
if importlib.util.find_spec("gmsh") is None:
    pip("gmsh")
try:
    import gmsh  # noqa: F401
except OSError:                       # the wheel needs libGLU and a few X libraries (missing on some images)
    if IN_COLAB:
        subprocess.run("apt-get install -y -qq libglu1-mesa libxcursor1 libxinerama1 libxft2 > /dev/null",
                       shell=True, check=True)

REPO_RAW = "https://raw.githubusercontent.com/PINNeAPPle-Labs/PINNeAPPle/main/"
try:
    from pinneapple_systems.process_components import joule_thomson, valve_cfd  # noqa: F401
except ImportError:
    import pinneapple_systems, pinneapple_veriphysics
    for pkg, rel in ((pinneapple_systems, "pinneapple_systems/process_components/joule_thomson.py"),
                     (pinneapple_systems, "pinneapple_systems/process_components/control_valve.py"),
                     (pinneapple_systems, "pinneapple_systems/process_components/valve_cfd.py"),
                     (pinneapple_veriphysics, "pinneapple_veriphysics/policy.py")):
        dest = Path(pkg.__path__[0]).parent / rel
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
        print("baixado do repositório:", rel)
    importlib.invalidate_caches()

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from pinneapple_systems.process_components.joule_thomson import (
    DEFAULT_GAS, EOS, isenthalpic_path, parse_valve_request, throttle, valve_joule_thomson_study)
import CoolProp
print("CoolProp", CoolProp.__version__, "| equações de estado:", ", ".join(EOS))

In [ ]:
# Parâmetros. SMOKE=True encurta as varreduras para o notebook rodar em segundos (o CI roda assim).
SMOKE = os.environ.get("PINNEAPPLE_NB_SMOKE", "0") == "1"
N_PATH = 11 if SMOKE else 41        # pontos do caminho isentálpico
N_SWEEP = 5 if SMOKE else 15        # pontos de cada varredura
OUT = Path("jt_valvula")
OUT.mkdir(exist_ok=True)

## 1. O caso, em palavras

O leitor de texto entende português e inglês: pressões (bar, barg, kPa, MPa, psi), temperaturas (°C, K, °F), Cv,
DN ou diâmetro, a composição e a temperatura que o CFD deu. O que não estiver no texto usa um padrão, e o
resultado diz qual foi o padrão usado. **Troque o texto pelo seu caso.**

In [ ]:
PEDIDO = ("Gás natural passando por uma válvula globo: pressão de entrada 70 bar, temperatura de entrada 25 °C, "
          "pressão de saída 20 bar, Cv 20, DN 100. Composição: metano 90 %, etano 5 %, propano 2 %, "
          "nitrogênio 2 %, CO2 1 %. A temperatura de saída do meu CFD é 24,6 °C.")

caso = parse_valve_request(PEDIDO)
print("lido do texto:  ", caso["found"])
print("valor padrão:   ", caso["defaulted"] or "nenhum")
print("caso:           ", caso["case"])
print("gás (fração molar):", caso["gas"])

## 2. Um estrangulamento, quatro equações de estado

`throttle` resolve a saída de um estrangulamento adiabático. Sem vazão, ele impõe $h_2 = h_1$. Com a vazão e a
área do tubo, ele conserva a entalpia de estagnação, $h_2 + v_2^2/2 = h_1 + v_1^2/2$; a diferença entre os dois é a
**correção cinética**, pequena nas velocidades de tubo.

In [ ]:
c, gas = caso["case"], caso["gas"]
P1, T1, P2 = c["P1_bar"] * 1e5, c["T1_C"] + 273.15, c["P2_bar"] * 1e5

print(f"{'equação de estado':<16} {'T2 (°C)':>9} {'ΔT (K)':>8} {'μJT entrada (K/bar)':>20} {'Z entrada':>10}")
for eos in EOS:
    r = throttle(gas, P1, T1, P2, eos=eos)
    print(f"{eos:<16} {r['T2_K'] - 273.15:9.2f} {r['dT_K']:8.2f} {r['mu_JT_inlet_K_per_bar']:20.3f} {r['Z1']:10.3f}")

O gás ideal dá ΔT = 0, exatamente. Neste caso, Peng-Robinson e SRK ficam a até 2,4 K do GERG-2008. O GERG-2008 é a
referência da ISO 20765 para gás natural; Peng-Robinson e SRK são as cúbicas que os códigos de CFD costumam
oferecer.

## 3. O caminho isentálpico

Ao longo da válvula, o estado anda sobre a isentálpica que passa pela entrada. O $\mu_{JT}$ cresce à medida que a
pressão cai (para estas condições), então o gás esfria cada vez mais por bar perdido.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for eos, cor in (("GERG-2008", "tab:green"), ("Peng-Robinson", "tab:orange"), ("SRK", "tab:purple")):
    p = isenthalpic_path(gas, P1, T1, P2, n=N_PATH, eos=eos)
    axs[0].plot(p["P"] / 1e5, p["T"] - 273.15, color=cor, label=eos)
    axs[1].plot(p["P"] / 1e5, p["mu_JT"] * 1e5, color=cor, label=eos)
axs[0].axhline(c["T1_C"], color="tab:red", ls="--", label="gás ideal")
for ax in axs:
    ax.invert_xaxis(); ax.set_xlabel("pressão (bar)"); ax.grid(alpha=0.3); ax.legend()
axs[0].set_ylabel("temperatura (°C)"); axs[0].set_title("T ao longo da isentálpica")
axs[1].set_ylabel("μ_JT (K/bar)"); axs[1].set_title("Coeficiente de Joule-Thomson")
plt.tight_layout(); plt.show()

## 4. O estudo completo, com o seu CFD

`valve_joule_thomson_study` junta tudo:

- a temperatura de saída por equação de estado, com a correção cinética;
- a vazão da válvula pela IEC 60534-2-1, com a checagem de escoamento bloqueado;
- o **ponto mais frio dentro do jato** (local, no estado sônico): é bem mais frio que a saída, e é ali que pode
  formar hidrato ou condensar;
- as checagens: a energia conservada, o flash contra a integral do $\mu_{JT}$, o limite de gás ideal, a fase na
  saída, o escoamento bloqueado, a dispersão entre as cúbicas e o **seu CFD**;
- a nota VeriPhysics e as figuras.

A temperatura do CFD veio do texto (`24,6 °C`). Você também pode passá-la direto, com `cfd_T2_C=...`.

In [ ]:
r = valve_joule_thomson_study(PEDIDO, out_dir=str(OUT))

o = r["outputs"]
print(f"saída (GERG-2008, com correção cinética): {o['T2_C']:.2f} °C  (ΔT = {o['dT_K']:+.2f} K)")
print(f"ponto mais frio no jato: {o['T_jet_coldest_C']:.1f} °C a {o['P_jet_sonic_bar']:.1f} bar")
if "cfd_error_K" in o:
    print(f"seu CFD está {o['cfd_error_K']:+.1f} K acima da referência")
print("\nnota VeriPhysics:", r["decision"]["verdict"])
print()
for ch in r["checks"]:
    print(f"[{ch['status']:>4}] {ch['code']}: {ch['detail']}")

In [ ]:
for f in r["files"]:
    display(Image(filename=str(OUT / f), width=900))

## 5. Sensibilidade

Quanto o gás esfria depende principalmente da queda de pressão e da temperatura de entrada; a composição pesa
menos. Aqui, o GERG-2008 com $h_2 = h_1$.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
P2s = np.linspace(5, c["P1_bar"] - 5, N_SWEEP)
axs[0].plot(P2s, [throttle(gas, P1, T1, p * 1e5)["dT_K"] for p in P2s], "o-")
axs[0].set_xlabel("pressão de saída (bar)"); axs[0].set_title(f"a partir de {c['P1_bar']:g} bar e {c['T1_C']:g} °C")

T1s = np.linspace(0, 60, N_SWEEP)
axs[1].plot(T1s, [throttle(gas, P1, t + 273.15, P2)["dT_K"] for t in T1s], "o-")
axs[1].set_xlabel("temperatura de entrada (°C)"); axs[1].set_title(f"{c['P1_bar']:g} → {c['P2_bar']:g} bar")

composicoes = {"metano puro": {"Methane": 1.0}, "gás do caso": gas,
               "mais pesado": {"Methane": 0.80, "Ethane": 0.10, "Propane": 0.05, "Nitrogen": 0.02,
                               "CarbonDioxide": 0.03}}
dts = [throttle(g, P1, T1, P2)["dT_K"] for g in composicoes.values()]
axs[2].bar(list(composicoes), dts, color=["tab:blue", "tab:green", "tab:orange"])
for i, v in enumerate(dts):
    axs[2].text(i, v, f"{v:.1f} K", ha="center", va="bottom")
axs[2].set_title("composição")
for ax in axs:
    ax.set_ylabel("ΔT (K)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. O que mudar no CFD

Se o seu CFD deu praticamente a mesma temperatura na saída e na entrada, verifique, nesta ordem:

1. **Equação de estado de gás real.** No OpenFOAM, em `constant/thermophysicalProperties`, troque
   `equationOfState perfectGas;` por `equationOfState PengRobinsonGas;` e informe `Tc`, `Pc`, `Vc` e `omega` (com
   uma mistura, use propriedades pseudocríticas ou o componente dominante). No Fluent, use *real-gas
   Peng-Robinson*; no CFX, *Redlich-Kwong* ou *Peng-Robinson*. Com gás ideal, o resfriamento é **zero** por
   construção.
2. **Equação da energia em entalpia total**, com o termo de trabalho de pressão e a energia cinética
   (`rhoSimpleFoam` ou `rhoPimpleFoam` com `energy sensibleEnthalpy`). Uma equação só de temperatura, ou sem esses
   termos, perde o efeito.
3. **Condições de contorno:** pressão total e temperatura total na entrada, pressão estática na saída e paredes
   adiabáticas. Paredes isotérmicas aquecem o gás de volta.
4. **Malha na vena contracta**, logo depois do obturador. O jato é muito mais frio (veja o "ponto mais frio no jato"
   acima); uma malha grossa ali mistura e mascara esse pico.
5. **Onde medir.** Compare a temperatura **média ponderada pela massa** numa seção a uns 10 diâmetros a jusante, e
   não um ponto no jato, com a saída isentálpica deste notebook.
6. **Convergência:** o resíduo da energia e o balanço de entalpia (entra = sai, com paredes adiabáticas).

Com o gás real, o CFD deve chegar perto da saída do GERG-2008 acima; a checagem `cfd.outlet_temperature` aceita 10 % da queda (2,5 K neste caso), a ordem da diferença entre as equações de estado. As seções 7 e 8 fazem exatamente isso: geram a geometria e rodam o CFD no OpenFOAM pelo PINNeAPPle, com gás ideal
e com Peng-Robinson.

## 7. A geometria da válvula, gerada com o PINNeAPPle

`ValveGeometry` descreve o caminho de estrangulamento de uma válvula globo, simplificado para um **obturador e
assento axissimétricos** em volta de uma haste. O gás sai do tubo de entrada pelo furo do assento, vira para fora
pela folga entre o assento e o obturador (o curso da válvula, *lift*), contorna a borda do obturador e se mistura no
tubo de saída. O corpo em S da válvula globo não entra: a perda de carga extra dele fica antes ou depois do
estrangulamento e não muda a temperatura de saída, que, num escoamento adiabático, é fixada pela entalpia de
estagnação.

Todas as medidas estão em mm; troque pelas da sua válvula. `mesh_valve` gera a malha com o gmsh, uma cunha de 5°
(o OpenFOAM resolve o problema como axissimétrico), refinada no assento e no obturador.

In [ ]:
from pinneapple_systems.process_components.valve_cfd import (ValveGeometry, geometry_figure, mesh_valve,
                                                             openfoam_available, valve_cfd_study)

geo = ValveGeometry(pipe_d=100, stem_d=16, seat_d=40, seat_len=10, plug_d=60, plug_len=15, lift=4,
                    upstream=150, downstream=500)
print("áreas (mm²):", {k: round(v * 1e6, 1) for k, v in geo.areas_m2().items()},
      "-> a folga (curtain) é a garganta")
display(Image(filename=geometry_figure(geo, str(OUT / "valve_geometry.png")), width=900))
malha = mesh_valve(geo, str(OUT / "malha"), h_fine_mm=1.0 if SMOKE else 0.4, h_coarse_mm=5.0 if SMOKE else 3.0)
print(f"malha: {malha['cells']} células (hexaedros), arquivos {malha['msh']} e {malha['stl']}")

## 8. O CFD no OpenFOAM, pelo PINNeAPPle

`valve_cfd_study` escreve o caso do OpenFOAM, roda e lê os resultados de volta, duas vezes: com **gás ideal**
(`perfectGas`) e com **Peng-Robinson** (`PengRobinsonGas`).

- Solver: `rhoPimpleFoam` com passo de tempo local até o regime permanente, `transonic` (o escoamento fica sônico na
  folga), k-ω SST.
- Gás: a mistura vira um fluido pseudo-puro. As constantes críticas e o fator acêntrico vêm pela regra de Kay, e o
  Cp de gás ideal pelo GERG-2008 (CoolProp), da composição informada.
- Contorno: pressão total e temperatura na entrada, pressão estática na saída, paredes adiabáticas.
- Leitura: temperatura de saída média ponderada pela vazão mássica, vazão de entrada e de saída, balanço de entalpia
  de estagnação e histórico de convergência.

As checagens comparam o CFD com o estrangulamento da biblioteca (GERG-2008 e Peng-Robinson) na vazão do próprio CFD,
e com o modelo de gás do caso, à mesma entalpia de estagnação: se a energia se conserva, os dois batem.

**No Colab**, a célula abaixo instala o OpenFOAM v1912, o pacote `openfoam` do Ubuntu (uns 3 minutos). Cada caso
leva uns 3 a 4 minutos em CPU, com 8 mil células e 4 mil iterações.

In [ ]:
RODAR_CFD = not SMOKE          # o CI roda o notebook sem o OpenFOAM
if RODAR_CFD and not openfoam_available():
    if IN_COLAB:
        subprocess.run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openfoam "
                       "> /tmp/apt_openfoam.log 2>&1", shell=True, check=True)
    print("OpenFOAM disponível:", openfoam_available())
RODAR_CFD = RODAR_CFD and openfoam_available()

In [ ]:
if RODAR_CFD:
    cfd = valve_cfd_study(str(OUT / "cfd"), P1_bar=c["P1_bar"], T1_C=c["T1_C"], P2_bar=c["P2_bar"],
                          composition=gas, geom=geo, iterations=4000, lang="pt")
    for eos, run in cfd["runs"].items():
        print(f"{eos:>13}: saída {run['T_out_C']:.2f} °C, vazão {run['mdot_out']:.3f} kg/s, "
              f"{run['cells']} células, {run['run_seconds']:.0f} s")
    for nome, ref in cfd["reference"].items():
        print(f"estrangulamento {nome} (na vazão do CFD): {ref['T2_C']:.2f} °C")
    print("\nVeriPhysics:", cfd["decision"]["verdict"])
    for ch in cfd["checks"]:
        print(f"[{ch['status']:>4}] {ch['code']}: {ch['detail']}")
    for f in cfd["files"]:
        display(Image(filename=str(OUT / "cfd" / f), width=1000))
else:
    print("CFD não rodado aqui (modo SMOKE ou sem OpenFOAM); veja os resultados medidos na célula abaixo.")

**Resultados medidos ao preparar este notebook** (OpenFOAM v1912, 1 núcleo de CPU; caso padrão: 70 → 20 bar, 25 °C, gás ilustrativo, a geometria acima):

| | temperatura de saída | vazão mássica | tempo de execução |
|---|---|---|---|
| CFD, gás ideal (`perfectGas`) | 24.30 °C | 5.888 kg/s | 132 s |
| CFD, Peng-Robinson (`PengRobinsonGas`) | -1.81 °C | 6.452 kg/s | 202 s |
| estrangulamento Peng-Robinson da biblioteca (mistura), na vazão do CFD | -2.57 °C | | |
| estrangulamento GERG-2008, na vazão do CFD | -0.20 °C | | |

Com gás ideal o CFD não esfria (-0.70 K, só o termo cinético), exatamente o sintoma do TCC; com Peng-Robinson esfria 26.8 K. Balanço de entalpia de estagnação: -3 J/kg (ideal) e +43 J/kg (Peng-Robinson); o CFD bate com o modelo de gás do próprio caso em 0.02 K e 0.00 K. Nota VeriPhysics: **APPROVED**.

Checagens: `cfd.ideal.converged` pass, `cfd.ideal.energy_balance` pass, `cfd.ideal.model_consistency` pass, `cfd.pr.converged` pass, `cfd.pr.energy_balance` pass, `cfd.pr.model_consistency` pass, `cfd.ideal.no_cooling` pass, `cfd.pr.vs_pr` pass, `cfd.pr.vs_gerg` pass.

## 9. O seu caso

Mude os valores e rode a célula. Ela usa os argumentos diretos em vez do texto.

In [ ]:
meu = valve_joule_thomson_study(
    P1_bar=70, T1_C=25, P2_bar=20,      # pressões (bar) e temperatura de entrada (°C)
    Cv=20, pipe_d_mm=100,               # Cv da válvula e diâmetro do tubo (mm)
    cfd_T2_C=24.6,                      # a temperatura de saída do seu CFD (°C); None se não tiver
    composition=dict(DEFAULT_GAS),      # frações molares (nomes do CoolProp: Methane, Ethane, ...)
    out_dir=str(OUT / "meu_caso"), lang="pt")
print(f"T2 = {meu['outputs']['T2_C']:.2f} °C, ΔT = {meu['outputs']['dT_K']:+.2f} K, "
      f"VeriPhysics {meu['decision']['verdict']}")

## Referências

- O. Kunz e W. Wagner (2012), *The GERG-2008 wide-range equation of state for natural gases and other mixtures*,
  J. Chem. Eng. Data 57, 3032–3091.
- IEC 60534-2-1 (2011), *Industrial-process control valves, Part 2-1: Flow capacity — sizing equations for fluid
  flow under installed conditions*.
- I. H. Bell et al. (2014), *Pure and pseudo-pure fluid thermophysical property evaluation and the open-source
  thermophysical property library CoolProp*, Ind. Eng. Chem. Res. 53, 2498–2508.
- OpenFOAM v1912 (ESI), `rhoPimpleFoam` com `PengRobinsonGas`: https://www.openfoam.com.
- O código: [`pinneapple_systems/process_components/valve_cfd.py`](https://github.com/PINNeAPPle-Labs/PINNeAPPle/blob/main/pinneapple_systems/process_components/valve_cfd.py) e [`pinneapple_systems/process_components/joule_thomson.py`](https://github.com/PINNeAPPle-Labs/PINNeAPPle/blob/main/pinneapple_systems/process_components/joule_thomson.py).